# 02 · Transfer learning

Una red entrenada en una tarea A aprende filtros (bordes, rayas, texturas) que sirven para otra
tarea B. Con pocos datos en B, reutilizarlos gana a entrenar desde cero. Lo vemos en pequeño: A =
4 orientaciones de rayas (3 000 ejemplos); B = 3 orientaciones *distintas* (22,5°, 82,5° y 142,5°),
con 10 ejemplos por clase y mucho más ruido.

In [1]:
import numpy as np

from common import set_seed, setup_style

setup_style()
rng = set_seed(42)
import torch
from torch import nn


class SmallCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
        )
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.head = nn.Linear(64, n_classes)

    def forward(self, x):
        return self.head(self.pool(self.features(x)))


def train(model, X, y, epochs=8, lr=1e-3, params=None):
    opt = torch.optim.Adam(params if params is not None else model.parameters(), lr=lr)
    gen = torch.Generator().manual_seed(42)
    for _ in range(epochs):
        model.train()
        for idx in torch.randperm(len(X), generator=gen).split(64):
            opt.zero_grad()
            nn.functional.cross_entropy(model(X[idx]), y[idx]).backward()
            opt.step()
    return model


@torch.no_grad()
def accuracy(model, X, y):
    model.eval()
    return (model(X).argmax(1) == y).float().mean().item()


def gratings(n, task, noise, size=32):
    yy, xx = np.mgrid[0:size, 0:size] / size
    X, y = np.empty((n, 1, size, size), np.float32), np.empty(n, np.int64)
    for i in range(n):
        label = rng.integers(4 if task == "A" else 3)
        # A: 0°, 45°, 90°, 135°; B: 22,5°, 82,5°, 142,5° (ninguna se vio en A)
        angle = label * np.pi / 4 if task == "A" else np.pi / 8 + label * np.pi / 3
        freq, phase = rng.uniform(3, 6), rng.uniform(0, 6)
        wave = np.sin(2 * np.pi * freq * (xx * np.cos(angle) + yy * np.sin(angle)) + phase)
        X[i, 0] = wave + rng.normal(0, noise, (size, size))
        y[i] = label
    return torch.tensor(X), torch.tensor(y)


torch.manual_seed(42)
XA, yA = gratings(3000, "A", noise=0.8)
XB_test, yB_test = gratings(600, "B", noise=2.5)
print(XA.shape, XB_test.shape)

torch.Size([3000, 1, 32, 32]) torch.Size([600, 1, 32, 32])


In [2]:
base = train(SmallCNN(4), XA, yA, epochs=6)
print(f"tarea A (orientación), exactitud en train: {accuracy(base, XA, yA):.2f}")

tarea A (orientación), exactitud en train: 1.00


Con 30 ejemplos el resultado depende mucho de cuáles tocaron: repetimos 5 veces con muestras
distintas de B y promediamos. El fine-tuning usa dos tasas de aprendizaje: 1e-3 para la cabeza nueva
(parte de cero) y 1e-4 para los filtros (solo se retocan).

In [3]:
import copy


def transfer(head_only):
    model = copy.deepcopy(base)
    model.head = nn.Linear(64, 3)
    if head_only:  # sonda lineal: filtros congelados
        for p in model.features.parameters():
            p.requires_grad_(False)
        return model, model.head.parameters()
    groups = [
        {"params": model.head.parameters()},
        {"params": model.features.parameters(), "lr": 1e-4},
    ]
    return model, groups


results = {"desde cero": [], "sonda lineal": [], "fine-tuning": []}
for rep in range(5):
    XB, yB = gratings(30, "B", noise=2.5)
    torch.manual_seed(rep)
    results["desde cero"].append(train(SmallCNN(3), XB, yB, epochs=40))
    for name, head_only in (("sonda lineal", True), ("fine-tuning", False)):
        model, params = transfer(head_only)
        results[name].append(train(model, XB, yB, epochs=40, params=params))

for name, models in results.items():
    accs = [accuracy(m, XB_test, yB_test) for m in models]
    print(f"{name:13s} exactitud en test B: {np.mean(accs):.2f} (mín. {min(accs):.2f})")

desde cero    exactitud en test B: 0.81 (mín. 0.65)
sonda lineal  exactitud en test B: 0.93 (mín. 0.70)


fine-tuning   exactitud en test B: 0.89 (mín. 0.68)


**Lectura.** Con 30 ejemplos ruidosos, la red desde cero tiene que aprender a la vez los
filtros y la decisión. Las transferidas reutilizan detectores de orientación de A, que sirven aunque
las orientaciones de B sean nuevas: la sonda solo aprende la decisión, y el fine-tuning además
retoca los filtros, lo que con tan pocos datos puede ayudar o sobreajustar. En el proyecto, A es
ImageNet (1,2 millones de fotos) y B son mazorcas: la pregunta es cuánto sirve un preentrenamiento
de un dominio tan distinto.